# Quantization from scratch
Post-training int8/int4 weight quantization of a NumPy MLP.

## 1. Train the fp32 model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from mlp import make_data, init_mlp, train, accuracy
from quant import quantize, dequantize, quantize_model, fp32_size
X, y = make_data(6000, 1); Xt, yt = make_data(3000, 2)
rng = np.random.default_rng(42)
p = init_mlp([20, 128, 64, 8], rng)
train(p, X, y, 40, 2e-3, 64, rng)
print('fp32 acc', accuracy(p, Xt, yt), 'size', fp32_size(p), 'bytes')

## 2. Quantize one tensor by hand

In [ ]:
w = p['W1']
for sym in (True, False):
    for pc in (False, True):
        q, s, z = quantize(w, 4, sym, pc)
        e = dequantize(q, s, z) - w
        print(f"int4 sym={sym!s:5} per_channel={pc!s:5} q range=[{q.min()}, {q.max()}] scales={s.size:3d} mse={np.mean(e**2):.2e}")

## 3. Accuracy vs bits

In [ ]:
for bits in (8, 4, 3, 2):
    accs = [accuracy(quantize_model(p, bits, sym, pc)[0], Xt, yt) for sym in (True, False) for pc in (False, True)]
    print(bits, 'bits  sym/pt, sym/pc, asym/pt, asym/pc:', [round(a, 4) for a in accs])

## 4. Full smoke run
Run `python run_smoke.py` from the repo root. It writes `results/`.